# Fox and hounds

Authors: Shelby Bagwell, Daniel Lillard, Kaden Ellingson, James Best

In [11]:
# Gameboard definition
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


class GameBoard:
    def __init__(self):
        # create 6x6 board
        # place fox and 4 hounds in starting positions
        # set first turn to fox
        self.board_size = 6
        self.board = self._make_board()
        self.current_player = "FOX"
        self.game_over = False

    def _make_board(self):
        # 6x6 board
        board = np.zeros((self.board_size, self.board_size))
        for r in range(self.board_size):
            for c in range(self.board_size):
                if (r + c) % 2 == 1:
                    board[r, c] = -1  # light square, all 0s are dark

        # Set up Initial Hounds
        board[5, 3] = 1
        board[4, 4] = 1
        board[5, 5] = 1
        board[3, 5] = 1

        # Set up initial fox
        board[0, 0] = 2  # top left corner

        return board

    def move_piece(self, start_pos, end_pos):
        # check if move is valid
        avail_moves = self.get_available_moves(self.current_player)
        if (start_pos, end_pos) not in avail_moves:
            print(
                f"ERROR: Player {self.current_player} - Invalid move: {start_pos} to {end_pos}"
            )
            return False

        # update board state
        piece = self.board[start_pos]
        self.board[end_pos] = piece
        self.board[start_pos] = 0  # spot is now empty

        # switch player turn
        winner = self.check_win()
        if winner:
            # do something
            pass
        # commenting this out for now, I want to
        # just move the fox around
        # if self.current_player == "FOX":
        #     self.current_player = "HOUNDS"
        # else:
        #     self.current_player = "FOX"

        return True

    def get_piece_moves(self, r, c):
        moves = []
        piece = self.board[r, c]

        # Fox logic
        if piece == 2:
            dirs = [(-1, -1), (-1, 1), (1, -1), (1, 1)]
        # Hound logic
        elif piece == 1:
            dirs = [(-1, 1), (-1, 1)]
        else:
            # something went wrong??
            print("ERROR: invalid piece type")
            return []

        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if (
                0 <= nr < self.board_size  # valid row
                and 0 <= nc < self.board_size  # valid column
                and (self.board[nr, nc] == 0  # space is unoccupied 
                     or self.board[nr, nc] == -1)  # light square
            ):
                moves.append((nr, nc))

        return moves

    def get_available_moves(self, player):
        # get all valid moves on board for specified player
        all_moves = []

        if self.current_player == "FOX":
            piece_val = 2
        else:
            piece_val = 1

        cur_coords = np.argwhere(self.board == piece_val)

        for r, c in cur_coords:
            piece_moves = self.get_piece_moves(r, c)
            for end_pos in piece_moves:
                all_moves.append(((r, c), end_pos))

        return all_moves

    def check_win(self):
        fox_pos = np.argwhere(self.board == 2)
        fox_r, fox_c = fox_pos[0]

        # is fox at the end?
        if fox_r == 5 and fox_c == 1:
            print("fox_r, fox_c", fox_r, fox_c)
            self.game_over = True
            winner = "FOX"
            return winner

        # can fox not move anymore?
        if self.current_player == "FOX":
            fox_moves = self.get_available_moves("FOX")
            if not fox_moves:
                self.game_over = True
                winner = "HOUNDS"

        # nobody wins yet
        return None

    def render_board(self):
        fig, ax = plt.subplots(figsize=(6, 6))

        cmap = ListedColormap(["black", "white"])
        board_display = np.zeros((self.board_size, self.board_size))
        for r in range(self.board_size):
            for c in range(self.board_size):
                board_display[r, c] = (r + c) % 2

        ax.imshow(board_display, cmap=cmap, interpolation="nearest")

        for r in range(self.board_size):
            for c in range(self.board_size):
                if self.board[r, c] == 1:
                    ax.scatter(
                        c,
                        r,
                        color="blue",
                        s=500,
                        edgecolors="black",
                        linewidth=2,
                    )
                elif self.board[r, c] == 2:
                    ax.scatter(
                        c,
                        r,
                        color="red",
                        s=500,
                        edgecolors="black",
                        linewidth=2,
                    )

        ax.set_xticks(np.arange(self.board_size))
        ax.set_yticks(np.arange(self.board_size))
        ax.set_title(f"Turn: {self.current_player}")
        plt.show()


In [12]:
import random   # for random move selection
def Fox_Random_AI(board):
    available_moves = board.get_available_moves("FOX")
    move = random.choice(available_moves)
    return move

Next two blocks are for the fox AI

In [ ]:
def dijkstra(board):
    fox_pos = np.argwhere(board == 2)
    fox_r, fox_c = fox_pos[0]
    # target is bottom right
    target = (5, 5)
    # create set of unvisited nodes
    unvisited = set()
    for r in range(6):
        for c in range(6):
            if board[r, c] == 0:  # only consider dark squares
                unvisited.add((r, c))
    unvisited.add((fox_r, fox_c)) # add fox position to unvisited
    # ensure that the target is in unvisited
    if target not in unvisited:
        print("Target position is not reachable.")
        return []
    # initialize distances
    distances = {pos: float('inf') for pos in unvisited}
    distances[(fox_r, fox_c)] = 0
    previous_nodes = {pos: None for pos in unvisited}
    while unvisited:
        current = min(unvisited, key=lambda pos: distances[pos])
        if current == target or distances[current] == float('inf'):
            break
        unvisited.remove(current)
        r, c = current
        neighbors = [(r + dr, c + dc) for dr, dc in [(-1, -1), (-1, 1), (1, -1), (1, 1)]]
        for nr, nc in neighbors:
            if (nr, nc) in unvisited:
                alt = distances[current] + 1
                if alt < distances[(nr, nc)]:
                    distances[(nr, nc)] = alt
                    previous_nodes[(nr, nc)] = current
    path = []
    current = target
    while current and current in previous_nodes:
        path.append(current)
        current = previous_nodes[current]
    path.reverse()
    return path

In [14]:
def Fox_Short_Path_AI(board):
    path = dijkstra(board.board)
    if len(path) > 1:
        next_move = path[1]  # the first element is the current position
        fox_pos = np.argwhere(board.board == 2)
        fox_r, fox_c = fox_pos[0]
        return ( (fox_r, fox_c), next_move )
    else:
        return Fox_Random_AI(board)

In [ ]:
# first configuration, random vs random
# turn count is defined as a fox move followed by a hound move
# or if the fox wins on its move, that counts as a turn
fox_win_count = 0
hound_win_count = 0
turn_count = 0
# opening up a csv to log results
with open("random_vs_random_results.csv", "w") as f:
    f.write("Game,Winner,Turns\n")
    for i in enumerate(range(100)):
        # create a game
        rand_vs_rand_board = GameBoard()
        while rand_vs_rand_board.check_win() is None:
            Fox_Random_AI(rand_vs_rand_board)
            # we can move the hounds with certainty they will not win
            # as the fox cannot move itself into a hole, and if the fox
            # is at the bottom, it cannot be trapped, and the hound move
            # will not cause a win
            Hounds_Random_AI(rand_vs_rand_board)
            turn_count += 1
        if rand_vs_rand_board.check_win() == "FOX":
            fox_win_count += 1
        if rand_vs_rand_board.check_win() == "HOUNDS":
            hound_win_count += 1
        f.write(f"{i[0]+1},{rand_vs_rand_board.check_win()},{turn_count}\n")
        turn_count = 0
print(f"Random Fox win count: {fox_win_count}, Random Hounds win count: {hound_win_count}")

NameError: name 'Hound_Random_AI' is not defined

In [ ]:
# second configuration, shortest path fox vs random hounds
fox_win_count = 0
hound_win_count = 0
turn_count = 0
with open("dijkstras_vs_random_results.csv", "w") as f:
    f.write("Game,Winner,Turns\n")
    for i in enumerate(range(100)):
        # create a game
        dijkstra_vs_rand_board = GameBoard()
        while dijkstra_vs_rand_board.check_win() is None:
            Fox_Short_Path_AI(dijkstra_vs_rand_board)
            Hound_Random_AI(dijkstra_vs_rand_board)
        if dijkstra_vs_rand_board.check_win() == "FOX":
            fox_win_count += 1
        if dijkstra_vs_rand_board.check_win() == "HOUNDS":
            hound_win_count += 1

        f.write(f"{i[0]+1},{dijkstra_vs_rand_board.check_win()},{turn_count}\n")
        turn_count = 0
print(f"Dijkstras Fox win count: {fox_win_count}, Random Hounds win count: {hound_win_count}")

In [ ]:
# Third configuration, random fox vs Minimax hounds
fox_win_count = 0
hound_win_count = 0
turn_count = 0
with open("random_vs_minimax_results.csv", "w") as f:
    f.write("Game,Winner,Turns\n")
    for i in enumerate(range(100)):
        # create a game
        rand_vs_minimax_board = GameBoard()
        while rand_vs_minimax_board.check_win() is None:
            Fox_Random_AI(rand_vs_minimax_board)
            Hounds_Minimax_AI(rand_vs_minimax_board)
        if rand_vs_minimax_board.check_win() == "FOX":
            fox_win_count += 1
        if rand_vs_minimax_board.check_win() == "HOUNDS":
            hound_win_count += 1

        f.write(f"{i[0]+1},{rand_vs_minimax_board.check_win()},{turn_count}\n")
        turn_count = 0
print(f"Random Fox win count: {fox_win_count}, Minimax Hounds win count: {hound_win_count}")

In [ ]:
# Fourth configuration, Dijkstras fox vs Minimax hounds
fox_win_count = 0
hound_win_count = 0
turn_count = 0
with open("Dijkstras_vs_Minimax_results.csv", "w") as f:
    f.write("Game,Winner,Turns\n")
    for i in enumerate(range(100)):
        # create a game
        dijkstra_vs_minimax_board = GameBoard()
        while dijkstra_vs_minimax_board.check_win() is None:
            Fox_Short_Path_AI(dijkstra_vs_minimax_board)
            Hounds_Minimax_AI(dijkstra_vs_minimax_board)
        if dijkstra_vs_minimax_board.check_win() == "FOX":
            fox_win_count += 1
        if dijkstra_vs_minimax_board.check_win() == "HOUNDS":
            hound_win_count += 1
        f.write(f"{i[0]+1},{dijkstra_vs_minimax_board.check_win()},{turn_count}\n")
        turn_count = 0
print(f"Dijkstras Fox win count: {fox_win_count}, Minimax Hounds win count: {hound_win_count}")